# Inflation différenciée en France: actualisation Insee 2026

Ce notebook reprend la logique du mémoire: appliquer des variations de prix identiques à des paniers de consommation différents. Il utilise exclusivement des données Insee: Budget de famille 2017 pour les paniers et IPC d'août 2026 pour les prix.

**Attention:** les résultats sont des calculs à paniers fixes, pas des indices catégoriels publiés par l'Insee.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.analyse import CATEGORY_SPECS, calculate_category, load_price_ratios

divisions, ratios, metadata = load_price_ratios()
all_results, all_details = [], []
for category, spec in CATEGORY_SPECS.items():
    result, detail = calculate_category(category, spec, ratios)
    all_results.append(result)
    all_details.append(detail)
results = pd.concat(all_results, ignore_index=True)
details = pd.concat(all_details, ignore_index=True)

## Lieu de résidence

In [ ]:
territoires = results.query("category == 'lieu_de_residence'").copy()
territoires[['group_label', 'modeled_inflation', 'difference_vs_modeled_total']].round(3)

In [ ]:
display(Image(filename=str(ROOT / 'outputs/figures/inflation_lieu_de_residence.png')))

## Décomposition de l'écart rural-Paris

In [ ]:
territory_detail = details.query("category == 'lieu_de_residence'")
contrib = territory_detail.pivot(index=['division', 'division_label'], columns='group_code', values='contribution_points')
contrib['ecart_rural_paris'] = contrib['0'] - contrib['4']
contrib[['ecart_rural_paris']].sort_values('ecart_rural_paris', ascending=False).round(3)

In [ ]:
display(Image(filename=str(ROOT / 'outputs/figures/contributions_ecart_rural_paris.png')))

## Autres catégories

In [ ]:
for category in CATEGORY_SPECS:
    print('\n', category.upper())
    display(results.query("category == @category and group_code != 'TOT'")
            [['group_label', 'modeled_inflation', 'difference_vs_modeled_total']]
            .sort_values('modeled_inflation', ascending=False)
            .round(3))

## Contrôles et limites

In [ ]:
pd.Series(metadata, name='valeur').to_frame()

Le panier moyen construit avec les dépenses BDF 2017 ne reproduit pas exactement l'IPC officiel, dont les pondérations sont actualisées chaque année. Cette différence n'est pas corrigée artificiellement: l'analyse porte sur les écarts entre groupes calculés avec la même méthode.

La division 12 de l'ancienne COICOP est raccordée aux divisions 12 et 13 de l'eCOICOP v2 avec les pondérations nationales de l'IPC 2026. Les paniers de 2017 ne captent ni les substitutions depuis 2017, ni les différences locales de niveau de prix, ni les évolutions de revenu.